<a href="https://colab.research.google.com/github/NadiaJeni/ArrayCategoriesTest/blob/main/FraudDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Run this first cell in Colab (installs packages)
!pip install --quiet lightgbm scikit-learn torch opacus matplotlib openpyxl


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.4/254.4 kB 3.2 MB/s eta 0:00:00


In [2]:
# Upload your dataset file (fraudTest.csv/xlsx). Run and choose file.
from google.colab import files
import shutil, os
print("Upload your dataset (fraudTest.csv or fraudTest.xlsx).")
uploaded = files.upload()
for fn in uploaded.keys():
    print("Uploaded:", fn)
    # move to working location
    shutil.move(fn, "/content/" + fn)
print("Files in /content:", os.listdir("/content")[:20])

Upload your dataset (fraudTest.csv or fraudTest.xlsx).


Saving fraudTest.csv.xlsx to fraudTest.csv.xlsx
Uploaded: fraudTest.csv.xlsx
Files in /content: ['.config', 'fraudTest.csv.xlsx', 'sample_data']


In [5]:
# ===== Cell 3: Pseudo-label generation (robust, copy-paste & run) =====
!pip install --quiet scikit-learn openpyxl

import os, json, joblib, numpy as np, pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split

OUT_DIR = "/content/pseudo_label_output"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------- PARAMETERS ----------
TOP_PCT = 0.0025
        # top 0.5% as pseudo positive (tuneable)
ISO_CONTAMINATION = 0.001  # expected fraction of anomalies for IsolationForest
USE_ANN_MODEL_IF_EXISTS = False  # set True ONLY if you have `model_global` loaded in this session
# --------------------------------

# ---------- helper: feature engineering (reuse your FE) ----------
def feature_engineering(df):
    df = df.copy()
    if 'transdatetrans_time' in df.columns and 'trans_date_trans_time' not in df.columns:
        df.rename(columns={'transdatetrans_time': 'trans_date_trans_time'}, inplace=True)
    if 'trans_date_trans_time' in df.columns:
        df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'], errors='coerce')
    if 'dob' in df.columns:
        df['dob'] = pd.to_datetime(df['dob'], errors='coerce')
    if 'trans_date_trans_time' in df.columns and 'dob' in df.columns:
        df['age'] = (df['trans_date_trans_time'] - df['dob']).dt.days / 365.25
        df['hour'] = df['trans_date_trans_time'].dt.hour.fillna(0).astype(int)
    else:
        df['hour'] = df.get('hour', 0)
        df['age'] = df.get('age', 40)
    # distance
    def haversine_row(r):
        try:
            from math import radians, sin, cos, sqrt, atan2
            if pd.isna(r.get('lat')) or pd.isna(r.get('long')) or pd.isna(r.get('merch_lat')) or pd.isna(r.get('merch_long')):
                return 0.0
            R = 6371
            lat1, lon1, lat2, lon2 = map(radians, [r['lat'], r['long'], r['merch_lat'], r['merch_long']])
            dlat = lat2 - lat1
            dlon = lon2 - lon1
            a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
            return 2 * R * atan2(sqrt(a), sqrt(1-a))
        except:
            return 0.0
    if set(['lat','long','merch_lat','merch_long']).issubset(df.columns):
        df['dist_km'] = df.apply(haversine_row, axis=1)
    else:
        df['dist_km'] = df.get('dist_km', 0.0)

    if 'amt' in df.columns:
        df['high_amt'] = (df['amt'] >= 1000).astype(int)
    else:
        df['amt'] = 0.0; df['high_amt'] = 0

    df['night_txn'] = ((df['hour'] >= 22) | (df['hour'] < 6)).astype(int)
    df['far_merchant'] = (df['dist_km'] > 50).astype(int)
    df['old_and_big_amt'] = ((df['age'] >= 65) & (df['amt'] >= 500)).astype(int)
    return df

# ---------- Load unlabeled dataset ----------
if 'X_new' in globals():
    df_new = X_new.copy()
    print("Using dataframe X_new from session.")
else:
    print("Please upload your unlabeled file (Excel or CSV). File picker will open...")
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded)==0:
        raise RuntimeError("No file uploaded.")
    fname = list(uploaded.keys())[0]
    print("Uploaded:", fname)
    if fname.lower().endswith(('.xlsx', '.xls')):
        df_new = pd.read_excel(fname)
    else:
        df_new = pd.read_csv(fname)

print("Raw shape:", df_new.shape)

# ---------- Feature engineering ----------
df_new = feature_engineering(df_new)
print("After feature engineering shape:", df_new.shape)

# ---------- Compute weak-rule score ----------
def compute_rule_score(df):
    s = pd.Series(0.0, index=df.index)
    # tune weights if needed
    if 'amt' in df.columns:
        s += (df['amt'] > 5000).astype(float) * 2.0
        s += (df['amt'] > 2000).astype(float) * 1.0
    if 'night_txn' in df.columns:
        s += (df['night_txn'] == 1).astype(float) * 1.0
    if 'far_merchant' in df.columns:
        s += (df['far_merchant'] == 1).astype(float) * 1.0
    if 'age' in df.columns:
        s += (((df['age'] < 18) | (df['age'] > 80)).astype(float)) * 0.8
    risky = set(['electronics','jewelry','luxury_goods','flight'])
    if 'category' in df.columns:
        s += df['category'].fillna("__NA__").astype(str).str.lower().isin(risky).astype(float) * 1.5
    return s

df_new['rule_score'] = compute_rule_score(df_new).fillna(0.0)
print("Rule score computed. Example distribution:")
print(df_new['rule_score'].describe())

# ---------- IsolationForest anomaly score ----------
numeric_features = ['amt','city_pop','age','dist_km','hour','high_amt','night_txn','far_merchant','old_and_big_amt']
numeric_features = [c for c in numeric_features if c in df_new.columns]
if len(numeric_features)==0:
    raise RuntimeError("No numeric features available for anomaly detection. Check your dataset or feature list.")

scaler_iso = StandardScaler()
X_iso = scaler_iso.fit_transform(df_new[numeric_features].fillna(0.0).values)

iso = IsolationForest(n_estimators=200, contamination=ISO_CONTAMINATION, random_state=42)
iso.fit(X_iso)
anom_score_raw = -iso.decision_function(X_iso)   # higher => more anomalous
df_new['anom_score'] = anom_score_raw
print("Anomaly scores computed. Example distribution:")
print(df_new['anom_score'].describe())

# ---------- ANN model scoring (robust) ----------
ann_prob = None
if USE_ANN_MODEL_IF_EXISTS and 'model_global' in globals():
    try:
        import torch
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model_global.to(device)
        model_global.eval()

        # try to load feature_cols from previous fed_output if present
        feat_cols = None
        try:
            fp = "/content/fed_output/feature_cols_reconstructed.json"
            if os.path.exists(fp):
                feat_cols = json.load(open(fp))
        except Exception:
            feat_cols = None

        if feat_cols is None:
            cand = ['amt','city_pop','age','dist_km','hour','high_amt','night_txn','far_merchant','old_and_big_amt']
            feat_cols = [c for c in cand if c in df_new.columns]

        if len(feat_cols)==0:
            print("Warning: no feature_cols available for ANN scoring. Skipping ANN scoring.")
        else:
            X_feats = df_new[feat_cols].fillna(0.0).values.astype('float32')
            import numpy as np
            probs = []
            batch = 8192
            with torch.no_grad():
                for i in range(0, len(X_feats), batch):
                    xb = torch.tensor(X_feats[i:i+batch]).to(device)
                    out = model_global(xb)
                    if isinstance(out, (tuple, list)):
                        logits = out[0]
                    else:
                        logits = out
                    logits = logits.cpu().numpy().ravel()
                    probs.append(1.0/(1.0+np.exp(-logits)))
            ann_prob = np.concatenate(probs)
            df_new['ann_prob'] = ann_prob
            print("ANN probabilities computed and added as 'ann_prob'.")
    except Exception as e:
        print("Warning: model_global exists but scoring failed:", e)
        ann_prob = None
else:
    print("No ANN model found in session or ANN usage disabled. Skipping ANN scoring.")

# ---------- Combine signals into meta_score ----------
signals = ['rule_score','anom_score']
if 'ann_prob' in df_new.columns:
    signals.append('ann_prob')

sc = MinMaxScaler()
normed = sc.fit_transform(df_new[signals].fillna(0.0).values)
# simple average of normalized signals
df_new['meta_score'] = normed.mean(axis=1)

print("Meta score computed. Example distribution:")
print(df_new['meta_score'].describe())

# ---------- Select top fraction as pseudo positives ----------
n_top = max(1, int(TOP_PCT * len(df_new)))
top_idx = np.argsort(df_new['meta_score'].values)[-n_top:]
df_new['pseudo_label'] = 0
df_new.loc[df_new.index[top_idx], 'pseudo_label'] = 1

print(f"Selected top {n_top} rows as pseudo positives (pseudo_label=1).")

# ---------- Export top candidates for human review ----------
top_for_review = df_new.loc[df_new.index[top_idx]].sort_values('meta_score', ascending=False)
top_csv = os.path.join(OUT_DIR, "top_candidates_for_review.csv")
top_for_review.to_csv(top_csv, index=False)
print("Saved top candidates for human review to:", top_csv)

# ---------- Export full dataset with pseudo_label ----------
full_csv = os.path.join(OUT_DIR, "full_with_pseudo_labels.csv")
df_new.to_csv(full_csv, index=False)
print("Saved full dataset with pseudo_label to:", full_csv)

# ---------- Summary ----------
print("\nSummary counts:")
print(df_new['pseudo_label'].value_counts())
print("\nFiles saved in", OUT_DIR)
print("Recommendation: manually review 'top_candidates_for_review.csv' (at least 200-2000 rows) to create a verified gold set before final supervised training.")


Please upload your unlabeled file (Excel or CSV). File picker will open...


Saving fraudTest.csv.xlsx to fraudTest.csv (2).xlsx
Uploaded: fraudTest.csv (2).xlsx
Raw shape: (555719, 23)
After feature engineering shape: (555719, 30)
Rule score computed. Example distribution:
count    555719.000000
mean          1.146400
std           0.644428
min           0.000000
25%           1.000000
50%           1.000000
75%           1.800000
max           5.000000
Name: rule_score, dtype: float64
Anomaly scores computed. Example distribution:
count    555719.000000
mean         -0.230404
std           0.059984
min          -0.318476
25%          -0.280316
50%          -0.238789
75%          -0.196010
max           0.072353
Name: anom_score, dtype: float64
No ANN model found in session or ANN usage disabled. Skipping ANN scoring.
Meta score computed. Example distribution:
count    555719.000000
mean          0.227313
std           0.104269
min           0.085174
25%           0.140617
50%           0.181964
75%           0.315852
max           0.968763
Name: meta_score, d

In [4]:
# == Compare pseudo_label vs original is_fraud ==
import pandas as pd, numpy as np, os
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score

PSEUDO_PATH = "/content/pseudo_label_output/full_with_pseudo_labels.csv"   # from Cell-3
OUT_DIR = "/content/pseudo_label_output/compare_reports"
os.makedirs(OUT_DIR, exist_ok=True)

print("Loading:", PSEUDO_PATH)
df = pd.read_csv(PSEUDO_PATH)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist()[:30])

# Check is_fraud exists
if 'is_fraud' not in df.columns:
    raise RuntimeError("No 'is_fraud' column found in dataset — cannot compare. Add true labels first.")

# Ensure types
df['pseudo_label'] = df['pseudo_label'].fillna(0).astype(int)
df['is_fraud'] = df['is_fraud'].fillna(0).astype(int)

y_true = df['is_fraud'].values
y_pred = df['pseudo_label'].values

# Basic counts
total = len(df)
n_true_pos = int(y_true.sum())
n_pseudo_pos = int(y_pred.sum())
print(f"Total rows: {total}")
print(f"True frauds (is_fraud=1): {n_true_pos}")
print(f"Pseudo positives (pseudo_label=1): {n_pseudo_pos}")

# Confusion matrix
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
print("\nConfusion matrix (tn, fp, fn, tp):", (tn, fp, fn, tp))

# Metrics (treat pseudo as prediction)
prec = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
print(f"\nPrecision (pseudo vs true): {prec:.4f}")
print(f"Recall (pseudo vs true):    {rec:.4f}")
print(f"F1 (pseudo vs true):        {f1:.4f}")

# If meta_score exists, compute ordering Precision@K and ROC/PR on meta_score
if 'meta_score' in df.columns:
    scores = df['meta_score'].values
    try:
        roc = roc_auc_score(y_true, scores)
        pr = average_precision_score(y_true, scores)
    except Exception as e:
        roc, pr = None, None
    print("\nMeta_score ROC AUC:", roc)
    print("Meta_score PR AUC:", pr)

    def precision_at_k_from_scores(y_true, scores, ks=[10,50,100,500,1000,2000]):
        idx = np.argsort(scores)[::-1]
        out = {}
        for k in ks:
            k_ = min(k, len(scores))
            out[k] = int(y_true[idx[:k_]].sum())/k_
        return out

    print("Precision@K (by meta_score):", precision_at_k_from_scores(y_true, scores))

# Save examples for manual inspection
# True Positives (TP), False Positives (FP), False Negatives (FN)
tp_df = df[(df['is_fraud']==1) & (df['pseudo_label']==1)].copy()
fp_df = df[(df['is_fraud']==0) & (df['pseudo_label']==1)].copy()
fn_df = df[(df['is_fraud']==1) & (df['pseudo_label']==0)].copy()

tp_path = os.path.join(OUT_DIR, "true_positives_pseudo.csv")
fp_path = os.path.join(OUT_DIR, "false_positives_pseudo.csv")
fn_path = os.path.join(OUT_DIR, "false_negatives_pseudo.csv")

tp_df.to_csv(tp_path, index=False)
fp_df.to_csv(fp_path, index=False)
fn_df.to_csv(fn_path, index=False)

print("\nSaved example files:")
print(" TP ->", tp_path, " (# rows:", len(tp_df), ")")
print(" FP ->", fp_path, " (# rows:", len(fp_df), ")")
print(" FN ->", fn_path, " (# rows:", len(fn_df), ")")

# Also save a small 'top_candidates' file (by meta_score) for review if present
if 'meta_score' in df.columns:
    topk = 500
    top_idx = np.argsort(df['meta_score'].values)[::-1][:topk]
    df.loc[top_idx].to_csv(os.path.join(OUT_DIR, f"top_{topk}_by_meta_for_review.csv"), index=False)
    print("Saved top", topk, "by meta_score for review.")

# Summary report (human readable)
report = {
    "total_rows": int(total),
    "true_positives_count": int(n_true_pos),
    "pseudo_positives_count": int(n_pseudo_pos),
    "confusion": {"tn":int(tn),"fp":int(fp),"fn":int(fn),"tp":int(tp)},
    "precision": float(prec),
    "recall": float(rec),
    "f1": float(f1),
    "meta_score_roc": float(roc) if 'meta_score' in df.columns and roc is not None else None,
    "meta_score_pr": float(pr) if 'meta_score' in df.columns and pr is not None else None
}
import json
with open(os.path.join(OUT_DIR, "pseudo_vs_true_report.json"), "w") as f:
    json.dump(report, f, indent=2)

print("\nWrote summary JSON ->", os.path.join(OUT_DIR, "pseudo_vs_true_report.json"))
print("All done. Inspect the CSV files in", OUT_DIR)


Loading: /content/pseudo_label_output/full_with_pseudo_labels.csv
Shape: (555719, 34)
Columns: ['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud', 'age', 'hour', 'dist_km', 'high_amt', 'night_txn', 'far_merchant', 'old_and_big_amt']
Total rows: 555719
True frauds (is_fraud=1): 2145
Pseudo positives (pseudo_label=1): 2778

Confusion matrix (tn, fp, fn, tp): (np.int64(551225), np.int64(2349), np.int64(1716), np.int64(429))

Precision (pseudo vs true): 0.1544
Recall (pseudo vs true):    0.2000
F1 (pseudo vs true):        0.1743

Meta_score ROC AUC: 0.9055769146763305
Meta_score PR AUC: 0.08816280751069734
Precision@K (by meta_score): {10: 0.0, 50: 0.0, 100: 0.15, 500: 0.252, 1000: 0.232, 2000: 0.179}

Saved example files:
 TP -> /content/pseudo_label_output/compare_reports/true_positives_pseudo.c

In [6]:
# == Compare new pseudo_label vs original is_fraud ==
import pandas as pd, numpy as np
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score

PSEUDO_PATH = "/content/pseudo_label_output/full_with_pseudo_labels.csv"
df = pd.read_csv(PSEUDO_PATH)

y_true = df['is_fraud'].astype(int).values
y_pred = df['pseudo_label'].astype(int).values
scores = df['meta_score'].values

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

print("Confusion:", (tn, fp, fn, tp))
print("Precision:", precision_score(y_true, y_pred, zero_division=0))
print("Recall:", recall_score(y_true, y_pred, zero_division=0))
print("F1:", f1_score(y_true, y_pred, zero_division=0))

# Precision@K with new meta_score ranking
idx = np.argsort(scores)[::-1]
for k in [10,50,100,500,1000,2000]:
    print(f"P@{k} =", int(y_true[idx[:k]].sum())/k)


Confusion: (np.int64(552478), np.int64(1096), np.int64(1852), np.int64(293))
Precision: 0.210943124550036
Recall: 0.1365967365967366
F1: 0.1658177702320317
P@10 = 0.0
P@50 = 0.0
P@100 = 0.15
P@500 = 0.252
P@1000 = 0.232
P@2000 = 0.179


In [7]:
# ===== Cell 4 (Updated for 20 Clients): Simulate Federated Clients (disk-backed) =====
# Requires: full_with_pseudo_labels.csv from Cell-3

import os, json
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# ---------- Config ----------
IN_PATH = "/content/pseudo_label_output/full_with_pseudo_labels.csv"
OUT_DIR = "/content/fed_sim"
N_CLIENTS = 20            # <<< UPDATED: 20 CLIENTS
STRATIFY_BY_PSEUDO = True
RANDOM_STATE = 42
CLIENT_PREFIX = "client_"
# --------------------------------

os.makedirs(OUT_DIR, exist_ok=True)
print("Loading pseudo-labeled file:", IN_PATH)
df = pd.read_csv(IN_PATH)
print("Total rows:", len(df))

# ---------- Feature column selection ----------
candidates = ['amt','city_pop','age','dist_km','hour',
              'high_amt','night_txn','far_merchant','old_and_big_amt']
freq_cols = [c for c in df.columns if c.endswith('_freq')]
feature_cols = [c for c in candidates if c in df.columns] + freq_cols
feature_cols = list(dict.fromkeys(feature_cols))

if len(feature_cols) == 0:
    raise RuntimeError("No feature columns detected. Re-run feature engineering (Cell-3).")

print("Feature columns used:", feature_cols)

# ---------- Preprocessing: StandardScaler ----------
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_all = df[feature_cols].fillna(0.0).astype(float).values
scaler.fit(X_all)

import joblib
joblib.dump({'feature_cols': feature_cols, 'scaler': scaler},
            os.path.join(OUT_DIR, "preproc_meta.pkl"))
print("Saved scaler + feature_cols ->", os.path.join(OUT_DIR, "preproc_meta.pkl"))

# ---------- Split into 20 clients ----------
if STRATIFY_BY_PSEUDO and 'pseudo_label' in df.columns and df['pseudo_label'].nunique() > 1:
    df_pos = df[df['pseudo_label']==1].sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    df_neg = df[df['pseudo_label']==0].sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    pos_chunks = np.array_split(df_pos, N_CLIENTS)
    neg_chunks = np.array_split(df_neg, N_CLIENTS)
    chunks = []
    for i in range(N_CLIENTS):
        chunks.append(pd.concat([pos_chunks[i], neg_chunks[i]], ignore_index=True)
                        .sample(frac=1, random_state=RANDOM_STATE))
else:
    df_shuf = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    chunks = np.array_split(df_shuf, N_CLIENTS)

# ---------- Save each client ----------
clients_info = {'n_clients': N_CLIENTS, 'clients': [], 'feature_cols': feature_cols}

for i, chunk in enumerate(chunks):
    client_id = f"{i:02d}"
    client_path = os.path.join(OUT_DIR, f"{CLIENT_PREFIX}{client_id}.csv")

    save_cols = feature_cols + ['pseudo_label']
    # include identifiers if present
    for idc in ['trans_id','id','transaction_id','tx_id']:
        if idc in chunk.columns and idc not in save_cols:
            save_cols.append(idc)

    # ensure missing feature columns exist (fill)
    for c in save_cols:
        if c not in chunk.columns:
            chunk[c] = 0

    chunk[save_cols].to_csv(client_path, index=False)
    size = len(chunk)
    pos = int(chunk['pseudo_label'].sum())

    clients_info['clients'].append({
        'client_idx': i,
        'path': client_path,
        'size': int(size),
        'pseudo_pos': pos
    })

    print(f"Saved client {i:02d}: rows={size}, pseudo_pos={pos}, path={client_path}")

# ---------- Save metadata ----------
with open(os.path.join(OUT_DIR, "clients_info.json"), "w") as f:
    json.dump(clients_info, f, indent=2)

print("\nSaved clients_info ->", os.path.join(OUT_DIR, "clients_info.json"))

# ---------- Diagnostics ----------
total_rows = sum([c['size'] for c in clients_info['clients']])
total_pos = sum([c['pseudo_pos'] for c in clients_info['clients']])

print("\nDiagnostics:")
print(" Total rows:", total_rows)
print(" Total pseudo positives:", total_pos)
print("\n Client distribution (index: size | pseudo_pos):")
for c in clients_info['clients']:
    print(f"  {c['client_idx']:02d}: {c['size']} | {c['pseudo_pos']}")

print("\nClients ready for FedAvg. NEXT: Run updated Cell-5 (FedAvg ANN training).")


Loading pseudo-labeled file: /content/pseudo_label_output/full_with_pseudo_labels.csv
Total rows: 555719
Feature columns used: ['amt', 'city_pop', 'age', 'dist_km', 'hour', 'high_amt', 'night_txn', 'far_merchant', 'old_and_big_amt']
Saved scaler + feature_cols -> /content/fed_sim/preproc_meta.pkl


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Saved client 00: rows=27787, pseudo_pos=70, path=/content/fed_sim/client_00.csv
Saved client 01: rows=27787, pseudo_pos=70, path=/content/fed_sim/client_01.csv
Saved client 02: rows=27787, pseudo_pos=70, path=/content/fed_sim/client_02.csv
Saved client 03: rows=27787, pseudo_pos=70, path=/content/fed_sim/client_03.csv
Saved client 04: rows=27787, pseudo_pos=70, path=/content/fed_sim/client_04.csv
Saved client 05: rows=27787, pseudo_pos=70, path=/content/fed_sim/client_05.csv
Saved client 06: rows=27787, pseudo_pos=70, path=/content/fed_sim/client_06.csv
Saved client 07: rows=27787, pseudo_pos=70, path=/content/fed_sim/client_07.csv
Saved client 08: rows=27787, pseudo_pos=70, path=/content/fed_sim/client_08.csv
Saved client 09: rows=27786, pseudo_pos=69, path=/content/fed_sim/client_09.csv
Saved client 10: rows=27785, pseudo_pos=69, path=/content/fed_sim/client_10.csv
Saved client 11: rows=27785, pseudo_pos=69, path=/content/fed_sim/client_11.csv
Saved client 12: rows=27785, pseudo_pos=

In [8]:
# ============================================================
#                CELL 5 — FEDERATED ANN TRAINING
#                (20 CLIENTS • FedAvg • No DP)
# ============================================================

import os, json, time
import numpy as np, pandas as pd, joblib
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

print("🔄 Loading client metadata...")

FED_DIR = "/content/fed_sim"
CLIENT_INFO = f"{FED_DIR}/clients_info.json"
PREPROC_META = f"{FED_DIR}/preproc_meta.pkl"

if not os.path.exists(CLIENT_INFO):
    raise RuntimeError("❌ clients_info.json missing — run Cell-4 first")

if not os.path.exists(PREPROC_META):
    raise RuntimeError("❌ preproc_meta.pkl missing — run Cell-4 first")

with open(CLIENT_INFO, "r") as f:
    info = json.load(f)

client_files = [c["path"] for c in info["clients"]]
n_clients = len(client_files)

pre = joblib.load(PREPROC_META)
feature_cols = pre["feature_cols"]
scaler = pre["scaler"]

print(f"✅ Found {n_clients} clients")
print("Feature columns:", feature_cols)

# -------------------- Model Definition --------------------
EMB_SIZE = 64

class SmallMLP(nn.Module):
    def __init__(self, n_in, emb_size=EMB_SIZE):
        super().__init__()
        self.fc1 = nn.Linear(n_in, 128)
        self.fc2 = nn.Linear(128, emb_size)
        self.head = nn.Linear(emb_size, 1)
        self.act = nn.ReLU()
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = self.act(self.fc1(x))
        x = self.dropout(x)
        emb = self.act(self.fc2(x))
        out = self.head(emb)
        return out.squeeze(-1), emb

# -------------------- Hyperparameters --------------------
N_ROUNDS = 12
LOCAL_EPOCHS = 2
BATCH_SIZE = 512
LR = 1e-3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# -------------------- Initialize Global Model --------------------
n_in = len(feature_cols)
global_model = SmallMLP(n_in).to(DEVICE)
global_state = {k: v.cpu() for k,v in global_model.state_dict().items()}

# -------------------- Local Training Function --------------------
def local_train(state_dict, file_path):
    df = pd.read_csv(file_path, usecols=feature_cols + ["pseudo_label"])

    X = df[feature_cols].fillna(0.0).values.astype("float32")
    X = scaler.transform(X)  # apply saved scaler

    y = df["pseudo_label"].astype("float32").values

    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32)
    )
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True)

    model = SmallMLP(n_in).to(DEVICE)
    model.load_state_dict(state_dict)

    crit = nn.BCEWithLogitsLoss()
    opt = optim.Adam(model.parameters(), lr=LR)

    model.train()
    for _ in range(LOCAL_EPOCHS):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logits, _ = model(xb)
            loss = crit(logits, yb)
            loss.backward()
            opt.step()

    return {k: v.cpu() for k, v in model.state_dict().items()}, len(df)

# -------------------- FedAvg Loop --------------------
print("\n🚀 Starting Federated Training...\n")

start = time.time()

for rnd in range(1, N_ROUNDS+1):
    print(f"----- ROUND {rnd}/{N_ROUNDS} -----")

    local_states = []
    sizes = []

    for client_path in client_files:
        st, sz = local_train(global_state, client_path)
        local_states.append(st)
        sizes.append(sz)

    # Weighted aggregation
    total = float(sum(sizes))
    new_state = {}

    for k in global_state.keys():
        agg = None
        for i, st in enumerate(local_states):
            weighted = st[k] * (sizes[i] / total)
            agg = weighted if agg is None else agg + weighted
        new_state[k] = agg

    global_state = new_state
    print(f"✔ Round {rnd} aggregated.\n")

# -------------------- Save Global Model --------------------
OUT_FED = "/content/fed_output"
os.makedirs(OUT_FED, exist_ok=True)

global_model.load_state_dict(global_state)
torch.save(global_model.state_dict(), f"{OUT_FED}/global_fed_ann.pt")

with open(f"{OUT_FED}/feature_cols_used.json", "w") as f:
    json.dump(feature_cols, f)

print("🎉 FedAvg Complete!")
print("Saved global model → /content/fed_output/global_fed_ann.pt")
print("Total training time (sec):", time.time() - start)


🔄 Loading client metadata...
✅ Found 20 clients
Feature columns: ['amt', 'city_pop', 'age', 'dist_km', 'hour', 'high_amt', 'night_txn', 'far_merchant', 'old_and_big_amt']
Using device: cpu

🚀 Starting Federated Training...

----- ROUND 1/12 -----
✔ Round 1 aggregated.

----- ROUND 2/12 -----
✔ Round 2 aggregated.

----- ROUND 3/12 -----
✔ Round 3 aggregated.

----- ROUND 4/12 -----
✔ Round 4 aggregated.

----- ROUND 5/12 -----
✔ Round 5 aggregated.

----- ROUND 6/12 -----
✔ Round 6 aggregated.

----- ROUND 7/12 -----
✔ Round 7 aggregated.

----- ROUND 8/12 -----
✔ Round 8 aggregated.

----- ROUND 9/12 -----
✔ Round 9 aggregated.

----- ROUND 10/12 -----
✔ Round 10 aggregated.

----- ROUND 11/12 -----
✔ Round 11 aggregated.

----- ROUND 12/12 -----
✔ Round 12 aggregated.

🎉 FedAvg Complete!
Saved global model → /content/fed_output/global_fed_ann.pt
Total training time (sec): 327.16202688217163


In [9]:
# ============================================================
#          CELL 6 — EXTRACT EMBEDDINGS & ANN PROB
# ============================================================

import os, json, joblib, numpy as np, pandas as pd
import torch
import torch.nn as nn

print("🔄 Loading model + preprocessing + dataset...")

# Paths
MODEL_PATH = "/content/fed_output/global_fed_ann.pt"
FEATURE_PATH = "/content/fed_output/feature_cols_used.json"
PREPROC_META = "/content/fed_sim/preproc_meta.pkl"
DATA_PATH = "/content/pseudo_label_output/full_with_pseudo_labels.csv"

EMB_DIR = "/content/embeddings"
os.makedirs(EMB_DIR, exist_ok=True)

# Load feature cols
with open(FEATURE_PATH, "r") as f:
    feature_cols = json.load(f)

# Load scaler
pre = joblib.load(PREPROC_META)
scaler = pre["scaler"]

# Load full dataset
df = pd.read_csv(DATA_PATH)
X = df[feature_cols].fillna(0).values.astype(np.float32)
X = scaler.transform(X).astype(np.float32)

# Define ANN (same architecture)
EMB_SIZE = 64

class SmallMLP(nn.Module):
    def __init__(self, n_in, emb_size=EMB_SIZE):
        super().__init__()
        self.fc1 = nn.Linear(n_in, 128)
        self.fc2 = nn.Linear(128, emb_size)
        self.head = nn.Linear(emb_size, 1)
        self.act = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
    def forward(self, x):
        x = self.act(self.fc1(x))
        x = self.dropout(x)
        emb = self.act(self.fc2(x))
        out = self.head(emb)
        return out.squeeze(-1), emb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SmallMLP(len(feature_cols), emb_size=EMB_SIZE).to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

# ------------ Batch Embedding Extraction ------------
def get_emb_and_prob(model, X_np, batch=4096):
    embs, probs = [], []
    with torch.no_grad():
        for i in range(0, len(X_np), batch):
            xb = torch.tensor(X_np[i:i+batch]).to(device)
            logits, emb = model(xb)
            prob = torch.sigmoid(logits).cpu().numpy()
            embs.append(emb.cpu().numpy())
            probs.append(prob)
    return np.vstack(embs), np.concatenate(probs)

print("⚙ Running ANN forward pass...")
emb_full, ann_prob_full = get_emb_and_prob(model, X)

print("Embeddings shape:", emb_full.shape)
print("ANN prob shape:", ann_prob_full.shape)

# Save embeddings
np.save(f"{EMB_DIR}/emb_full.npy", emb_full)

# Add ANN prob to dataset
df["ann_prob"] = ann_prob_full
df.to_csv(f"{EMB_DIR}/full_with_ann_probs.csv", index=False)

print("✅ Saved:")
print(" - emb_full.npy")
print(" - full_with_ann_probs.csv")
print("📌 Proceed to Cell-7: Train VST (LightGBM)")


🔄 Loading model + preprocessing + dataset...
⚙ Running ANN forward pass...
Embeddings shape: (555719, 64)
ANN prob shape: (555719,)
✅ Saved:
 - emb_full.npy
 - full_with_ann_probs.csv
📌 Proceed to Cell-7: Train VST (LightGBM)


In [11]:
# ===== Cell 7 (FIXED): Train LightGBM (VST) on ANN embeddings =====
import os, joblib, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score

EMB_DIR = "/content/embeddings"
emb = np.load(f"{EMB_DIR}/emb_full.npy")
df = pd.read_csv(f"{EMB_DIR}/full_with_ann_probs.csv")

y = df['pseudo_label'].astype(int).values

# Split
if y.sum() > 1:
    X_tr, X_val, y_tr, y_val = train_test_split(
        emb, y, test_size=0.2, random_state=42, stratify=y
    )
else:
    X_tr, X_val, y_tr, y_val = emb, emb, y, y

# LightGBM params
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.03,
    'n_estimators': 1000,
    'verbosity': -1
}

# imbalance handling
if y_tr.sum() > 0:
    params['scale_pos_weight'] = float((len(y_tr) - y_tr.sum()) / max(1.0, y_tr.sum()))

print("Training LightGBM with params:", params)

# FIX: use callbacks instead of early_stopping_rounds
callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=False),
    lgb.log_evaluation(period=0)
]

gbm = lgb.LGBMClassifier(**params)
gbm.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=callbacks
)

OUT_GBM = f"{EMB_DIR}/vst_gbm_on_emb.pkl"
joblib.dump(gbm, OUT_GBM)
print("Saved VST GBM ->", OUT_GBM)

# Validation metrics (pseudo-label)
val_prob = gbm.predict_proba(X_val)[:, 1]
print("Val ROC AUC (pseudo):", roc_auc_score(y_val, val_prob))
print("Val PR  AUC (pseudo):", average_precision_score(y_val, val_prob))

# Save GBM prob for meta learner
gbm_prob_full = gbm.predict_proba(emb)[:, 1]
df['gbm_prob'] = gbm_prob_full
df.to_csv(f"{EMB_DIR}/full_with_ann_probs_and_gbm.csv", index=False)
print("Saved -> full_with_ann_probs_and_gbm.csv")


Training LightGBM with params: {'objective': 'binary', 'metric': 'auc', 'boosting_type': 'gbdt', 'num_leaves': 31, 'learning_rate': 0.03, 'n_estimators': 1000, 'verbosity': -1, 'scale_pos_weight': 399.15751575157515}
Saved VST GBM -> /content/embeddings/vst_gbm_on_emb.pkl


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Val ROC AUC (pseudo): 0.99597412431392
Val PR  AUC (pseudo): 0.8773634523327549


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Saved -> full_with_ann_probs_and_gbm.csv


In [12]:
# ===== Run Meta + Final Eval =====
import os, joblib, numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

EMB_DIR = "/content/embeddings"
OUT_DIR = "/content/ensemble_output"
os.makedirs(OUT_DIR, exist_ok=True)

csv_in = os.path.join(EMB_DIR, "full_with_ann_probs_and_gbm.csv")
df = pd.read_csv(csv_in)

# meta inputs
ann_prob = df['ann_prob'].values
gbm_prob = df['gbm_prob'].values
X_meta = np.vstack([ann_prob, gbm_prob]).T
y_meta = df['pseudo_label'].astype(int).values

# train meta
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X_meta, y_meta, test_size=0.2, random_state=42, stratify=y_meta)
meta = LogisticRegression(max_iter=2000)
meta.fit(X_tr, y_tr)
joblib.dump(meta, os.path.join(EMB_DIR, "meta_lr.pkl"))
print("Saved meta ->", os.path.join(EMB_DIR, "meta_lr.pkl"))

# apply meta on full data
meta_full_prob = meta.predict_proba(X_meta)[:,1]
df['meta_prob'] = meta_full_prob
out_csv = os.path.join(OUT_DIR, "full_with_meta_probs.csv")
df.to_csv(out_csv, index=False)
print("Saved ensemble output ->", out_csv)

# Evaluate on true labels if available
if 'is_fraud' in df.columns:
    y_true = df['is_fraud'].astype(int).values
    print("\nFinal evaluation on TRUE labels:")
    print("ROC AUC:", roc_auc_score(y_true, meta_full_prob))
    print("PR AUC:", average_precision_score(y_true, meta_full_prob))
    y_pred_05 = (meta_full_prob >= 0.5).astype(int)
    print("\nClassification report (threshold=0.5):")
    print(classification_report(y_true, y_pred_05, digits=4, zero_division=0))

    # Precision@K
    idx = np.argsort(meta_full_prob)[::-1]
    for k in [10,50,100,500,1000,2000]:
        k_ = min(k, len(idx))
        print(f"Precision@{k} = {int(y_true[idx[:k_]].sum())/k_:.4f}")
else:
    print("No 'is_fraud' column present — final eval skipped.")


Saved meta -> /content/embeddings/meta_lr.pkl
Saved ensemble output -> /content/ensemble_output/full_with_meta_probs.csv

Final evaluation on TRUE labels:
ROC AUC: 0.7328154622747578
PR AUC: 0.08723796711065143

Classification report (threshold=0.5):
              precision    recall  f1-score   support

           0     0.9967    0.9982    0.9975    553574
           1     0.2506    0.1566    0.1928      2145

    accuracy                         0.9949    555719
   macro avg     0.6236    0.5774    0.5951    555719
weighted avg     0.9939    0.9949    0.9944    555719

Precision@10 = 0.0000
Precision@50 = 0.0000
Precision@100 = 0.0900
Precision@500 = 0.3400
Precision@1000 = 0.3120
Precision@2000 = 0.1815


In [13]:
import numpy as np
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("/content/ensemble_output/full_with_meta_probs.csv")

ann_prob = df['ann_prob'].values
gbm_prob = df['gbm_prob'].values
meta_score = df['meta_score'].values  # from pseudo-label part

X_meta = np.vstack([ann_prob, gbm_prob]).T
y_meta = df['pseudo_label'].astype(int).values

# weights: high meta_score → more weight
w = meta_score ** 2

from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te, w_tr, w_te = train_test_split(
    X_meta, y_meta, w, test_size=0.2, random_state=42, stratify=y_meta)

meta_w = LogisticRegression(max_iter=3000)
meta_w.fit(X_tr, y_tr, sample_weight=w_tr)

df['meta_prob_w'] = meta_w.predict_proba(X_meta)[:,1]
df.to_csv("/content/ensemble_output/full_with_meta_probs_weighted.csv", index=False)

print("Weighted meta model saved.")


Weighted meta model saved.


In [14]:
# Evaluate weighted meta:
meta_scores = df['meta_prob_w'].values
y_true = df['is_fraud'].astype(int).values
idx = np.argsort(meta_scores)[::-1]

for k in [500, 1000, 2000]:
    print(f"P@{k}:", int(y_true[idx[:k]].sum())/k)


P@500: 0.34
P@1000: 0.312
P@2000: 0.1815


In [15]:
!zip -r /content/full_pipeline_outputs.zip /content/pseudo_label_output /content/fed_sim /content/fed_output /content/embeddings /content/ensemble_output


  adding: content/pseudo_label_output/ (stored 0%)
  adding: content/pseudo_label_output/compare_reports/ (stored 0%)
  adding: content/pseudo_label_output/compare_reports/true_positives_pseudo.csv (deflated 63%)
  adding: content/pseudo_label_output/compare_reports/false_negatives_pseudo.csv (deflated 68%)
  adding: content/pseudo_label_output/compare_reports/pseudo_vs_true_report.json (deflated 42%)
  adding: content/pseudo_label_output/compare_reports/top_500_by_meta_for_review.csv (deflated 56%)
  adding: content/pseudo_label_output/compare_reports/false_positives_pseudo.csv (deflated 61%)
  adding: content/pseudo_label_output/full_with_pseudo_labels.csv (deflated 56%)
  adding: content/pseudo_label_output/top_candidates_for_review.csv (deflated 59%)
  adding: content/fed_sim/ (stored 0%)
  adding: content/fed_sim/client_01.csv (deflated 64%)
  adding: content/fed_sim/client_18.csv (deflated 64%)
  adding: content/fed_sim/client_15.csv (deflated 64%)
  adding: content/fed_sim/clien

In [16]:
from google.colab import files
files.download('/content/full_pipeline_outputs.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>